In [ ]:
import sys
sys.path.insert(0, "..")

import pystac
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import pickle
from datetime import datetime
import rioxarray as rxr
from scipy.stats import randint, uniform

# --- RF ---
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold
from sklearn.model_selection import RandomizedSearchCV

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from helpers.shared import time_match


In [ ]:
from config import *

# -- Catalog --------------------------------------------------------------------------------------------
CATALOG                 = load_catalog()
INPUT_ASSET             = "train-stack"

# --- Bands ---

# --- Output Options ---
CACHE_DIR = Path("../out/rf/")

# --- Algorithm Parameters ---

METHOD = XGBRegressor

PARAMS = {
    RandomForestRegressor: CACHE_DIR / "hyperparam_search_RandomForestRegressor_20260428-123806.json",
    XGBRegressor: CACHE_DIR / "hyperparam_search_XGBRegressor_20260428-120837.json",
}

PARAM_DIST = {
    RandomForestRegressor: dict(
        n_estimators      = randint(100, 600),
        max_features      = ["sqrt", "log2", 0.3, 0.5],
        min_samples_leaf  = randint(1, 20),
        max_depth         = [None, 10, 20, 30],
    ),
    XGBRegressor: dict(
        n_estimators      = randint(200, 800),
        max_depth         = [3, 4, 5, 6, 8],
        learning_rate     = uniform(0.01, 0.19),
        subsample         = uniform(0.6, 0.4),
        colsample_bytree  = uniform(0.5, 0.5),
        min_child_weight  = randint(1, 20),
        reg_alpha         = uniform(0, 1),
        reg_lambda        = uniform(0.5, 2),
    ),
}


In [ ]:
stack_col = CATALOG.get_child("stacks")
train_stack = stack_col.assets[INPUT_ASSET]
sampled_df = pd.read_parquet(train_stack.href)

In [ ]:
# --- Features ---
DROP = {"x", "y", "block_id", "block_x", "block_y", "elev"}
FEATURE_COLS = [c for c in sampled_df.columns if c not in DROP]
RESPONSE = "elev"
# ROUTER = "sg_percov"
GROUP = "block_id"

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

class DiagMulticol:
    """
    variance inflation factor (VIF), condition numbers (CN), variance decomposition proportions (VDP)
    """

    def __init__(self, X, feature_names = None):
        self.X = np.array(X, dtype = np.float32)
        _, k = self.X.shape
        self.k = k
        self.feature_names = feature_names or [f"feat_{i}" for i in range(k)]
        self._fit()

    # ============================================================================
    # 1. Fit
    # ============================================================================

    def _fit(self):
        X, k = self.X, self.k

        # VIF
        self.vif_ = np.array([variance_inflation_factor(X, j) for j in range(k)])
        self.tolerance_ = 1.0 / self.vif_

        # Correlation matrix: Z^T Z
        corr = np.corrcoef(X.T)

        # Eigendecomposition
        eigenvalues, eigenvectors = np.linalg.eigh(corr)
        order = np.argsort(eigenvalues)[::-1]
        self.eigenvalues_ = eigenvalues[order]
        self.eigenvectors_ = eigenvectors[:, order]

        # Condition numbers: KS = sqrt(lambda_max, lambda_s); condition number = max(KS)
        eps = np.finfo(float).eps * self.eigenvalues_.max() * self.k
        ev_clipped = np.clip(self.eigenvalues_, eps, None)
        self.condition_indices_ = np.sqrt(self.eigenvalues_.max() / ev_clipped)
        self.condition_number_ = float(np.nanmax(self.condition_indices_))

        # VDP: phi_js = u^2_js / lamda_s / epsilon_s (u ^ 2_js / lambda_s)
        # rows = predictors j, cols = condition indices s
        phi = self.eigenvectors_ ** 2 / self.eigenvalues_[np.newaxis, :]
        self.vdp_ = phi / phi.sum(axis = 1, keepdims = True)

    # ============================================================================
    # 2. Tables
    # ============================================================================

    def vif_table(self):
        return pd.DataFrame(
            {"VIF": np.round(self.vif_, 3), "Tolerance": np.round(self.tolerance_, 3)}, index=self.feature_names,
        )
    
    def collinearity_table(self):
        """Rows = condition indices (asc), columns = predictors. Values = VDP."""
        df = pd.DataFrame(
            np.round(self.vdp_.T, 3),   # (k_ci × k_predictors)
            columns=self.feature_names,
        )
        df.insert(0, "CI", np.round(self.condition_indices_, 3))
        df.insert(0, "Eigenvalue", np.round(self.eigenvalues_,       3))
        return df

    # ============================================================================
    # 3. Summary
    # ============================================================================

    # Kim 2019:
    # 1. If the variance inflation factor and tolerance are greater than 5 to 10 and lower than 0.1 to 0.2, respectively, multicollinearity exists.
    # 2. A condition number between 10 and 30 indicates the presence of multicollinearity and when a value is larger than 30, the multicollinearity is regarded as strong
    # 3. When two or more VDPs, which correspond to a common condition index higher than 10 to 30, are higher than 0.8 to 0.9, their associated explanatory variables are multicollinear

    def summary(self, vif_thresh=5, vif_tolerance_thresh = 0.1, ci_thresh=10, vdp_thresh=0.8):
        print(f"Thresholds: VIF > {vif_thresh}, Tolerance < {vif_tolerance_thresh}, CI > {ci_thresh}, VDP > {vdp_thresh}\n")
        cn_status = (
            f"Max CN = {self.condition_number_:.1f}; strong collinearity present"
            if self.condition_number_ > ci_thresh
            else f"Max CN = {self.condition_number_:.1f}; OK"
        )
        
        print(cn_status + "\n") 
        
        print("\n==================== VIF: ====================")
        print(self.vif_table().to_string())
        flagged_vif = [f for f, v in zip(self.feature_names, self.vif_) if v > vif_thresh]
        if flagged_vif:
            if flagged_vif: print(f" ! VIF > {vif_thresh} (Tolerance < {vif_tolerance_thresh:.2f}): {flagged_vif}")


        print("\n==================== Collinearity Diagnostics ====================")
        print(self.collinearity_table().to_string(index=False))

        high_ci = np.where(self.condition_indices_ > ci_thresh)[0]
        for s in high_ci:
            collinear = [self.feature_names[j] for j in range(self.k) if self.vdp_[j, s] > vdp_thresh]
            if len(collinear) >= 2:
                print(f" ! CI = {self.condition_indices_[s]:.1f}: collinear -> {collinear}")
    



# Feature Selection

In [ ]:
X = sampled_df[FEATURE_COLS].values
y = sampled_df[RESPONSE].values
# router = sampled_df[ROUTER].values
blocks = sampled_df[GROUP].values

In [ ]:
mc = DiagMulticol(X, feature_names = FEATURE_COLS)
mc.summary()

In [ ]:
# --- Stage 1 domain pruning ---

DROP_FEATS = (
    {'B', 'G', 'Y', 'R'} # drop in favour of ln bands
    # {f for f in FEATURE_COLS if f.startswith('log_')} # redundant with normal bands
    | {'a_B', 'a_G', 'a_R', 'Kd_G', 'zSD'} # collinear with Kd_G; Kd_G more interpretable
    # | {f for f in FEATURE_COLS if f.startswith('nd_') and f != 'nd_G_R'} # keep only normalised difference
    | {'Y'} # collinear with red
)

KEPT_FEATURES = [f for f in FEATURE_COLS if f not in DROP_FEATS]

X = sampled_df[KEPT_FEATURES].values

In [ ]:
mc = DiagMulticol(X, feature_names = KEPT_FEATURES)
mc.summary()

In [ ]:
# import joblib
# import json

# SEARCH_CACHE = Path(f"../out/rf/hyperparam_search_{METHOD.__name__}_{datetime.now().strftime('%Y%m%d-%H%M%S')}.json")

# search = RandomizedSearchCV(
#     estimator           = METHOD(**{"n_jobs": 1, "random_state": SEED}),
#     param_distributions = PARAM_DIST[METHOD],
#     n_iter              = 40,
#     cv                  = GroupKFold(n_splits=N_FOLDS),
#     scoring             = "r2",
#     n_jobs              = -1,
#     random_state        = SEED,
#     verbose             = 2,
# )
# search.fit(X, y, groups=blocks)

# results = {
#     "best_params"  : search.best_params_,
#     "best_score"   : search.best_score_,
#     "method"       : METHOD.__name__,
#     "n_iter"       : 40,
#     "n_features"   : len(KEPT_FEATURES),
#     "features"     : KEPT_FEATURES,
#     "sample_n"     : len(X),
#     "timestamp"    : datetime.now().isoformat(),
#     "cv_results"   : pd.DataFrame(search.cv_results_)[
#                         ["rank_test_score", "mean_test_score", "std_test_score", "params"]
#                      ].sort_values("rank_test_score").to_dict(orient="records"),
# }

# SEARCH_CACHE.write_text(json.dumps(results, indent=2))

# print(f"Cached: {SEARCH_CACHE}")

# print("Best params:", search.best_params_)
# print(f"Best CV R2:  {search.best_score_:.3f}")


In [ ]:
import json
best_params = json.loads(PARAMS[METHOD].read_text())["best_params"]

NOTE = None

In [ ]:
# for params in tqdm(ParameterSampler(PARAM_DIST, n_iter = 10, random_state = SEED)):
#     kfold   = GroupKFold(n_splits=N_FOLDS)
#     results = []

#     for fold, (tr, val) in enumerate(tqdm(kfold.split(X, y, blocks), total=N_FOLDS)):
#         m = METHOD(**params)
#         m.fit(X[tr], y[tr])
#         preds = m.predict(X[val])

#         results.append({
#             "fold"       : fold,
#             "mae"        : mean_absolute_error(y[val], preds),
#             "r2"         : r2_score(y[val], preds),
#             "rmse"       : np.sqrt(mean_squared_error(y[val], preds)),
#             "n_train"    : len(tr),
#             "n_val"      : len(val),
#             "modeller"   : METHOD.__name__,
#             "params"     : params,
#             "predictors" : FEATURE_COLS,
#             "note"       : NOTE,
#         })

#         tqdm.write(f"Fold {fold} | MAE={results[-1]['mae']:.3f} | RMSE={results[-1]['rmse']:.3f} | R2={results[-1]['r2']:.3f}")
    
#     cv_df = pd.DataFrame(results)
#     tqdm.write(f"\nMean | MAE={cv_df['mae'].mean():.3f} | RMSE = {cv_df['rmse'].mean():.3f} | R2={cv_df['r2'].mean():.3f}")
#     fname = datetime.now().strftime("kfold-%Y%m%d-%H%M%S.pkl")
#     out_path = CACHE_DIR / fname
#     with open(out_path, "wb") as f:
#         pickle.dump(cv_df, f)

In [ ]:
kfold   = GroupKFold(n_splits=N_FOLDS)
results = []

for fold, (tr, val) in enumerate(tqdm(kfold.split(X, y, blocks), total=N_FOLDS)):
    m = METHOD(**best_params)
    m.fit(X[tr], y[tr])
    preds = m.predict(X[val])

    results.append({
        "fold"       : fold,
        "mae"        : mean_absolute_error(y[val], preds),
        "r2"         : r2_score(y[val], preds),
        "rmse"       : np.sqrt(mean_squared_error(y[val], preds)),
        "n_train"    : len(tr),
        "n_val"      : len(val),
        "modeller"   : METHOD.__name__,
        "params"     : best_params,
        "predictors" : KEPT_FEATURES,
        "note"       : NOTE,
    })

    tqdm.write(f"Fold {fold} | MAE = {results[-1]['mae']:.3f} | RMSE = {results[-1]['rmse']:.3f} | R2 = {results[-1]['r2']:.3f}")

In [ ]:
# kfold = GroupKFold(n_splits = N_FOLDS)
# results = []

# for fold, (tr, val) in enumerate(tqdm(kfold.split(X, y, blocks), total = N_FOLDS)):
#     m = MoE(modeller = METHOD)
#     m.fit(X[tr], y[tr], router[tr])
#     preds = m.predict(X[val], router[val])

#     results.append({
#     "fold"          : fold,
#     "mae"           : mean_absolute_error(y[val], preds),
#     "r2"            : r2_score(y[val], preds),
#     "rmse"          : np.sqrt(mean_squared_error(y[val], preds)),
#     "n_train"       : len(tr),
#     "n_val"         : len(val),
#     "modeller"      : m.modeller.__name__,
#     "params"        : m.params,
#     "predictors"    : FEATURE_COLS,
#     "note"          : NOTE,
#     })

#     tqdm.write(f"Fold {fold} | MAE={results[-1]['mae']:.3f} | RMSE = {results[-1]['rmse']:.3f} | R2={results[-1]['r2']:.3f}")

In [ ]:
# for pv, rf in m.experts_.items():
#     imp = pd.Series(rf.feature_importances_, index=FEATURE_COLS).nlargest(5)
#     tqdm.write(f"\nExpert {int(pv)}")
#     tqdm.write(imp.to_string())

In [ ]:
cv_df = pd.DataFrame(results)
display(cv_df[["fold", "mae", "r2", "predictors"]].round(3))
tqdm.write(f"\nMean | MAE={cv_df['mae'].mean():.3f} | RMSE = {cv_df['rmse'].mean():.3f} | R2={cv_df['r2'].mean():.3f}")


In [ ]:
fname = datetime.now().strftime("kfold-%Y%m%d-%H%M%S.pkl")
out_path = CACHE_DIR / fname
with open(out_path, "wb") as f:
    pickle.dump(cv_df, f)
tqdm.write(f"Saved: {out_path}")

In [ ]:
pkl_files = sorted((CACHE_DIR).glob("kfold-*.pkl"))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for pkl in pkl_files:
    with open(pkl, "rb") as f:
        df = pickle.load(f)
    label = pkl.stem
    axes[0].plot(df["fold"], df["mae"], marker="o", label=label)
    axes[1].plot(df["fold"], df["r2"],  marker="o", label=label)

axes[0].set(title="MAE per fold", xlabel="Fold", ylabel="MAE")
axes[1].set(title="R2 per fold", xlabel="Fold", ylabel="R2")

for ax in axes:
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


# Final model

In [ ]:
m_med = XGBRegressor(**best_params)
m_med.fit(X, y)

q_params = {**best_params, "objective": "reg:quantileerror"}

m_low = XGBRegressor(**q_params, quantile_alpha=0.1)
m_low.fit(X, y)

m_high = XGBRegressor(**q_params, quantile_alpha=0.9)
m_high.fit(X, y)

model_path = CACHE_DIR / f"model-{METHOD.__name__}-final.pkl"
with open(model_path, "wb") as f:
    pickle.dump({"model": m_med, "lower": m_low, "upper": m_high, "features": KEPT_FEATURES, "params": best_params}, f)

In [ ]:
pd.Series(m_med.feature_importances_, index=KEPT_FEATURES).sort_values(ascending=False)


In [ ]:
from plotnine import geom_bar, coord_flip, scale_x_discrete, theme, ggplot, aes, labs, element_text, ggsave

imp = pd.Series(m_med.feature_importances_, index=KEPT_FEATURES).sort_values(ascending=False)
imp_df = imp.reset_index()
imp_df.columns = ["feature", "importance"]
imp_df["feature"] = pd.Categorical(imp_df["feature"], categories=imp_df["feature"][::-1], ordered=True)

p = (
    ggplot(imp_df, aes("feature", "importance"))
    + geom_bar(stat="identity", fill="tomato", alpha=0.85)
    + coord_flip()
    + labs(x="", y="Feature importance (gain)", title="")
    + theme(
        figure_size=(7.34/2.54, 3.5),
        text=element_text(size=7),
        axis_text=element_text(size=6),
        axis_title=element_text(size=7),
    )
)

ggsave(p, "../out/figs/feature_importance.png", dpi=300)

# Val

In [ ]:
# sd8_items = list(CATALOG.get_child("sd8-imagery").get_items())
# bathy_items = list(CATALOG.get_child("bathymetry").get_items())

# print(bathy_items)

# pred_item = next(i for i in sd8_items if "2025" in i.id)
# val_item = next(i for i in bathy_items if i.id == "validation-bathy")

In [ ]:
# from sklearn.metrics import root_mean_squared_error

# pred_da = rxr.open_rasterio(pred_item.assets["bathy-pred"].href, chunks=CHUNKS, masked=True)
# val_da  = rxr.open_rasterio(val_item.assets["validation-bathy"].href, chunks=CHUNKS, masked=True)
# std_da  = rxr.open_rasterio("../out/pred/bathy-std-sd8-20250610.tif", chunks=CHUNKS, masked=True)

# pred = pred_da.squeeze(drop=True).values.ravel()
# val  = val_da.squeeze(drop=True).values.ravel()
# std  = std_da.squeeze(drop=True).values.ravel()

# pred_min, pred_max = np.nanmin(pred), np.nanmax(pred)

# base_mask = (
#     np.isfinite(pred) & np.isfinite(val) & np.isfinite(std)
#     & (val >= pred_min) & (val <= pred_max)
# )

# thresholds = np.nanpercentile(std[base_mask], [100, 90, 75, 50])

# print(f"{'STD thresh':>12} {'n':>8} {'RMSE':>8} {'Bias':>8} {'R²':>8} {'% kept':>8}")
# for thresh in thresholds:
#     m = base_mask & (std <= thresh)
#     p, v = pred[m], val[m]
#     rmse = root_mean_squared_error(v, p)
#     bias = (p - v).mean()
#     r2   = np.corrcoef(v, p)[0, 1] ** 2
#     pct  = 100 * m.sum() / base_mask.sum()
#     print(f"{thresh:>12.3f} {m.sum():>8,} {rmse:>8.3f} {bias:>+8.3f} {r2:>8.3f} {pct:>7.1f}%")
